In [10]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
try:
    from ipywidgets import interact, IntSlider
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    from ipywidgets import interact, IntSlider

# Ensure paths are correct for your local structure
sys.path.insert(0, '.')
sys.path.insert(0, 'baseframework/')

from src.densoinn import DenSOINN
from baseframework.datahandler import load_data

# ==========================================
# 1. VISUALIZATION HELPERS
# ==========================================

def _draw_nodes_and_edges(ax, nodes, edges, node_colors, default_color='black',
                           highlight_nodes=None, highlight_color=None, highlight_size=180):
    """
    Shared low-level renderer: draws topological edges then neural nodes on
    a given axis, colored per node_colors = {node_id: color}.
    highlight_nodes (optional) get drawn larger/with a distinct marker, e.g.
    to mark density-peak cluster centers.
    """
    drawn_edges = set()
    for src_id, targets in edges.items():
        if src_id not in nodes:
            continue
        c1 = nodes[src_id]['centroid']
        color = node_colors.get(src_id, default_color)
        for tgt_id in targets:
            if tgt_id not in nodes:
                continue
            edge_tuple = tuple(sorted((src_id, tgt_id)))
            if edge_tuple not in drawn_edges:
                c2 = nodes[tgt_id]['centroid']
                ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color=color,
                        linewidth=1.6, alpha=0.75, zorder=2)
                drawn_edges.add(edge_tuple)

    highlight_nodes = highlight_nodes or set()
    for node_id, node_data in nodes.items():
        centroid = node_data['centroid']
        color = node_colors.get(node_id, default_color)
        if node_id in highlight_nodes:
            ax.scatter(centroid[0], centroid[1], color=highlight_color or color,
                       marker='*', edgecolor='black', linewidth=0.8,
                       s=highlight_size, zorder=4)
        else:
            ax.scatter(centroid[0], centroid[1], color=color, marker='X',
                       edgecolor='black', linewidth=0.5, s=70, zorder=3)


def _style_axis(ax, title):
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_aspect('equal', adjustable='datalim')
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])


def plot_densoinn_stages(X_raw, nodes, edges, components, centers, macro_clusters,
                          batch_name="Current"):
    """
    Renders the four key stages of one DenSOINN offline-clustering pass,
    side by side:
      1. Raw network    - raw data + nodes/edges, no clustering info yet
      2. Components     - nodes colored by weakly-connected component
      3. Density peaks   - same components, with chosen cluster centers starred
      4. Macro-clusters  - final labels after cascading, the algorithm's output
    """
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    cmap = plt.get_cmap('tab20')

    # ---- Panel 1: raw network, no clustering yet ----
    ax = axes[0]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.6, zorder=1)
    _draw_nodes_and_edges(ax, nodes, edges, node_colors={}, default_color='dimgrey')
    _style_axis(ax, f"1. [Online] Network (raw data + nodes)\n{len(nodes)} nodes")

    # ---- Panel 2: connected components ----
    ax = axes[1]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.4, zorder=1)
    comp_color = {}
    for i, comp in enumerate(components):
        c = cmap(i % 20)
        for nid in comp:
            comp_color[nid] = c
    _draw_nodes_and_edges(ax, nodes, edges, node_colors=comp_color)
    _style_axis(ax, f"2. [Offline] Connected components\n{len(components)} component(s)")

    # ---- Panel 3: density peaks / centers per component ----
    ax = axes[2]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.4, zorder=1)
    _draw_nodes_and_edges(ax, nodes, edges, node_colors=comp_color,
                          highlight_nodes=centers, highlight_color='red', highlight_size=220)
    _style_axis(ax, f"3. [Offline] Density-peak centers\n{len(centers)} center(s) found")

    # ---- Panel 4: final macro-clusters ----
    ax = axes[3]
    if len(X_raw) > 0:
        ax.scatter(X_raw[:, 0], X_raw[:, 1], c='gainsboro', s=12, alpha=0.4, zorder=1)
    unique_mcs = sorted(set(macro_clusters.values()))
    mc_color_map = {mc_id: cmap(i % 20) for i, mc_id in enumerate(unique_mcs)}
    macro_colors = {nid: mc_color_map.get(lbl, 'grey') for nid, lbl in macro_clusters.items()}
    _draw_nodes_and_edges(ax, nodes, edges, node_colors=macro_colors)
    _style_axis(ax, f"4. [Offline] Macro-clusters (final output)\n{len(unique_mcs)} cluster(s)")

    # Shared legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Raw Data', markerfacecolor='gainsboro', markersize=8),
        Line2D([0], [0], marker='X', color='w', label='Neural Node', markerfacecolor=cmap(0), markeredgecolor='black', markersize=9),
        Line2D([0], [0], marker='*', color='w', label='Cluster Center', markerfacecolor='red', markeredgecolor='black', markersize=13),
        Line2D([0], [0], color=cmap(0), lw=1.6, label='Topological Edge'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(f"DenSOINN Pipeline - {batch_name}", fontsize=15, fontweight='bold', y=1.03)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()


# ==========================================
# 2. SETUP PARAMETERS & LOAD REAL DATA
# ==========================================
dataset_name = 'complex9'
batch_size = 300

# DenSOINN Hyperparameters (Tuned for complex9 based on your earlier discussions)
gamma = 0.95           # Edge decay speed
stream_speed = 200     # Denoising frequency
epsilon = 0.5          # Denoising intensity
max_nodes = 1000       # Network capacity limit
alpha = 8         # Offline peak sensitivity multiplier
intrinsic_dim = 2      # Dimensionality for density estimation

print(f"Loading '{dataset_name}' data...")
# DenSOINN's self-adaptive metric handles unnormalized data natively, but we follow the runner's scaling default
X, y = load_data(dataset_name, scale=True)

densoinn = DenSOINN(
    gamma=gamma,
    stream_speed=stream_speed,
    epsilon=epsilon,
    max_nodes=max_nodes,
    alpha=alpha,
    intrinsic_dim=intrinsic_dim
)

num_batches = len(X) // batch_size
stream_history = []
all_X = []

print(f"Processing {num_batches} batches...")

# ==========================================
# 3. RUN ALGORITHM & CAPTURE STATE
# ==========================================
for i in range(num_batches + 1):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(X))
    batch = X[start_idx:end_idx]

    if len(batch) == 0:
        break

    timestamps = np.arange(start_idx, end_idx)

    # Run Layer 1 (Online Phase)
    densoinn.learn(batch, timestamps)
    all_X.extend(batch)

    # --- STATE EXTRACTION ---
    # 1. Deep copy the Neural Nodes
    current_nodes = {}
    for n_id, node in densoinn.online_phase.network.nodes.items():
        current_nodes[n_id] = {'centroid': node.w.copy()}

    # 2. Extract the Topological Edges (Weakly connected)
    current_edges = {}
    for src, tgts in densoinn.online_phase.network.edges.items():
        current_edges[src] = list(tgts.keys())

    # 3. Trigger Layer 2 (Offline Phase) with full stage breakdown:
    # components, density-peak centers, and final macro-cluster labels.
    summaries = densoinn.online_phase.get_summary()
    stages = densoinn.offline_phase.predict_summaries_with_stages(summaries)

    # Save the frame
    stream_history.append({
        'batch_name': f'Batch {i} (t={timestamps[-1]})',
        'X_raw': np.array(all_X),
        'nodes': current_nodes,
        'edges': current_edges,
        'components': stages['components'],
        'centers': stages['centers'],
        'macro_clusters': stages['labels'],
    })

print("Stream processing complete! Ready for visualization.")

# ==========================================
# 4. INTERACTIVE RENDERER
# ==========================================
def view_real_stream(batch_index):
    state = stream_history[batch_index]
    plot_densoinn_stages(
        X_raw=state['X_raw'],
        nodes=state['nodes'],
        edges=state['edges'],
        components=state['components'],
        centers=state['centers'],
        macro_clusters=state['macro_clusters'],
        batch_name=state['batch_name']
    )

# Launch the interactive slider
interact(view_real_stream, batch_index=IntSlider(min=0, max=len(stream_history)-1, step=1, value=0, description='Time Step:'))

Loading 'complex9' data...
Processing 10 batches...
Stream processing complete! Ready for visualization.


interactive(children=(IntSlider(value=0, description='Time Step:', max=10), Output()), _dom_classes=('widget-i…

<function __main__.view_real_stream(batch_index)>